In [8]:
import time
import random
import keyboard
import pygame

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns
import gymnasium as gym

In [ ]:
env = gym.make("FrozenLake-v1",
               desc = None,
               map_name = "4x4",
               is_slippery = False)

num_states = env.observation_space.n
num_actions = env.action_space.n

transitions = env.unwrapped.P
display('transient=', transitions)


In [ ]:
v = np.zeros(num_states)
pi = np.ones([num_states, num_actions]) * 0.25

theta = 1e-3
gamma = 0.95
count = 0

def policy_evaluation(v, pi, transitions, theta, gamma):
    while True:
        delta = 0
        for s in range(num_states):
            v_s = v[s]
            v[s] = sum([pi[s, a] * sum([p * (r + gamma * v[s_])
                                         for p, s_, r, _ in transitions[s][a]])
                        for a in range(num_actions)])
            delta = max(delta, abs(v_s - v[s]))
        if delta < theta:
            break
    return v

def policy_improvement(v, pi, transitions, gamma):
    policy_stable = True
    for s in range(num_states):
        old_action = np.argmax(pi[s])
        q = np.zeros(num_actions)
        for a in range(num_actions):
            q[a] = sum([transitions[s][a][p] * (r + gamma * v[s_])
                        for p, s_, r, _ in transitions[s][a]])
        best_action = np.argmax(q)
        if old_action != best_action:
            policy_stable = False
        pi[s] = np.zeros(num_actions)
        pi[s][best_action] = 1.0
    return pi, policy_stable